In [1]:
import numpy as np
from numba import njit

@njit
def mean_annualized(arr):
    n = arr.shape[0]
    return arr.sum(axis=0) / n * 252

@njit
def cov_annualized(arr):
    n, d = arr.shape
    mean = np.zeros(d)
    for j in range(d):
        for i in range(n):
            mean[j] += arr[i, j]
        mean[j] /= n

    centered = np.zeros((n, d))
    for i in range(n):
        for j in range(d):
            centered[i, j] = arr[i, j] - mean[j]

    cov = np.zeros((d, d))
    for i in range(d):
        for j in range(d):
            for k in range(n):
                cov[i, j] += centered[k, i] * centered[k, j]
            cov[i, j] /= (n - 1)

    return cov * 252


In [3]:
import pandas as pd

# 模擬資料 (1000 rows x 10 columns)
np.random.seed(0)
X = np.random.randn(1000, 10)

# Pandas DataFrame 版本
df = pd.DataFrame(X)


In [5]:
# 先跑一次 Numba，確保編譯完成
mean_annualized(X)
cov_annualized(X)


array([[ 2.60606330e+02,  5.40844650e+00,  7.78641458e+00,
         5.90699069e+00, -7.08013361e+00,  3.69254509e+00,
         7.14263301e+00,  2.15384253e+00, -6.17010842e+00,
        -2.87939760e+00],
       [ 5.40844650e+00,  2.43350328e+02, -4.52453847e+00,
         3.33126278e+00, -6.63702546e-01, -1.51133102e+01,
        -9.41541112e-01,  3.46011875e-01,  2.64931661e+00,
        -5.03227311e+00],
       [ 7.78641458e+00, -4.52453847e+00,  2.51929373e+02,
        -1.11874437e+00, -1.07343845e+01, -1.11622130e+00,
        -4.93050273e+00, -5.26655034e+00, -3.10212820e+00,
        -3.18377675e-01],
       [ 5.90699069e+00,  3.33126278e+00, -1.11874437e+00,
         2.39511484e+02,  6.43659224e+00, -1.13570927e-01,
        -1.71367844e+01, -1.22636975e+01,  1.26946279e+01,
        -5.41834431e+00],
       [-7.08013361e+00, -6.63702546e-01, -1.07343845e+01,
         6.43659224e+00,  2.47675679e+02,  2.41322905e-01,
         5.52801517e+00, -1.65129334e+01, -5.62365007e+00,
         1.

In [11]:
import time

start = time.time()
mu_pandas = df.mean() * 252
cov_pandas = df.cov() * 252
end = time.time()
print(f" Pandas Time: {end - start:.6f}s")


 Pandas Time: 0.018940s


In [13]:
start = time.time()
mu_numba = mean_annualized(X)
cov_numba = cov_annualized(X)
end = time.time()
print(f" Numba Time (after compile): {end - start:.6f}s")


 Numba Time (after compile): 0.003312s
